In [ ]:
import sys, os
# switch to repo root so imports work
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
from minigrad import *
from graphviz import Digraph

In [ ]:
dot = Digraph(graph_attr={'rankdir': 'LR'})
dot.node('B', shape='record')
dot.node(name='A', label = '{a | data: 4.23}', shape='record')
dot.node('C')
dot.edge('A','B')
dot.edge('C','B')
dot

In [ ]:
def trace(root):
    nodes = set()
    edges = set()

    # use DFS, DAG so keep going till all edges explored.
    nodes.add(root)
    for child in root._prev:
        edges.add((child, root))
        n, e = trace(child)
        nodes.update(n)
        edges.update(e)

    return nodes, edges


def draw_dot(root):
    dot = Digraph(
        graph_attr={'rankdir':'LR'}
    )
    
    nodes, edges = trace(root)

    for n in nodes:
        dot.node(name=str(id(n)), label="{ %s | data: %.4f | grad: %.4f}" % (n.label, n.data, n.grad), shape='record')
        # if op node, create one
        if n._op != '':
            dot.node(name = str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    for a, b in edges:
        dot.edge(str(id(a)), str(id(b)) + b._op)

    return dot

In [ ]:
a = Value(1.1) ; a.label = 'a'
b = Value(2.0) ; b.label = 'b'
c = b - a ; c.label = 'c'
e = c.exp(); e.label = 'e'
d = e**3 ; d.label = 'd'

d.backward()
x = draw_dot(d)
# x.render("node_viz.png", format='png')
x

In [ ]:
# testing grads
h = 0.0000001

a = Value(1.1) + h; a.label = 'a'
b = Value(2.0) ; b.label = 'b'
c = b - a; c.label = 'c'
e = c.exp(); e.label = 'e'
d1 = e**3 ; d.label = 'd'
print((d1.data - d.data) / h)

In [ ]:
from nn_tensor import MLP, Layer
import numpy as np

In [ ]:
 ## training
xs = [
    [0,0],
    [0,1],
    [1,1],
    [1,0]
]

ys = np.asarray([[1],[1],[1],[1]])

m = MLP(2, [2, 1])

In [ ]:
y_preds = m(xs)
y_preds

In [ ]:
h = 0.01
loss_vals = []
params = m.parameters()
for epoch in range(10000):
    y_preds = m(xs)
    loss = y_preds - ys
    loss = (loss*loss).sum()

    loss_vals.append(loss.data)

    # remember to zero grad!
    for p in params:
        p.grad = np.zeros_like(p.grad)

    loss.backward()
    for p in params:
        p.data -= p.grad * h

    if epoch % 1000 == 0:
        print(f'epoch {epoch}, loss={loss.data}')

print(loss)
print(y_preds)

In [ ]:
from matplotlib import pyplot as plt
plt.plot(loss_vals)
plt.xlim(left=0)
plt.ylim(bottom=0)
plt.show()

In [ ]:
for p in params:
    print(f"Val: {p.data}, Grad: {p.grad}")